In [ ]:
import torch
import numpy as np

def predict_autoregressive(model, initial_seq, dest_lat, dest_lon, scaler, max_steps, distance_threshold=0.1, device='cuda'):
    model.eval()
    input_seq = initial_seq.clone().to(device)
    all_preds_mu = []

    # 목적지 좌표 정규화
    dest_scaled = scaler.transform([[dest_lat, dest_lon, 0, 0]])[0]
    dest_lat_scaled, dest_lon_scaled = dest_scaled[0], dest_scaled[1]
    
    # 초기 상태 시퀀스도 넣어주기
    for step in range(max_steps):
        with torch.no_grad():
            mu = model(input_seq)
            pred_mu_np = mu.squeeze(1).cpu().numpy().flatten()  # (4,)
            
        all_preds_mu.append(pred_mu_np)

        # 예측된 lat/lon이 목적지에 도달했는지 판정
        pred_mu_denorm = scaler.inverse_transform(pred_mu_np.reshape(1, -1))[0]
        pred_lat, pred_lon = pred_mu_denorm[:2]

        if step % 10 == 0:
            print(f"[Step {step+1}] Predicted: ({pred_lat:.5f}, {pred_lon:.5f}) | "
                  f"Target: ({dest_lat:.5f}, {dest_lon:.5f}) | "
                  f"ΔLat: {abs(pred_lat - dest_lat):.5f}, ΔLon: {abs(pred_lon - dest_lon):.5f}")

        if abs(pred_lat - dest_lat) < distance_threshold and abs(pred_lon - dest_lon) < distance_threshold:
            print(f"🚢 목적지 도달 - Step: {step + 1}, {int(step/12)} 시간 {int((step%12)*5)} 분 소요")
            break

        # 시퀀스 슬라이딩
        input_seq_np = input_seq.squeeze(0).cpu().numpy()  # shape: (T, 11)

        # 다음 입력 feature 구성
        next_input_features = pred_mu_denorm  # [lat, lon, sog, cog]
        next_input_features_scaled = scaler.transform(next_input_features.reshape(1, -1))[0]

        # Δlat, Δlon
        delta_lat = dest_lat - pred_lat
        delta_lon = dest_lon - pred_lon

        # 거리
        distance = np.sqrt(delta_lat ** 2 + delta_lon ** 2)

        # 총 7차원 next input vector
        next_input_array = np.concatenate([
            next_input_features_scaled,            # 4개
            [dest_lat_scaled, dest_lon_scaled],    # 2개
            [distance],                            # 1개
        ])

        # 시퀀스 업데이트
        new_seq = np.vstack([input_seq_np[1:], next_input_array.reshape(1, -1)])  # shape: (T, 11)
        input_seq = torch.tensor(new_seq, dtype=torch.float32).unsqueeze(0).to(device)

    return np.stack(all_preds_mu)

In [ ]:
from math import atan2, sqrt, degrees
from math import pi, sin, cos
# 초기 시퀀스
pre = AISPreprocessor(data_dir='rou', input_seq_len=12, output_seq_len=1)
df = pd.read_csv('rou/yu_route_1_202003.csv', encoding='cp949', parse_dates=['일시'])
df = pre._preprocess_single_file(df)
initial_seq = df.iloc[:12]

# MinMaxScaler 수동 설정 ----------------------------------------------------------
# 정규화 범위 설정
lat_range = (33.0, 38.0)
lon_range = (124.0, 132.0)
sog_range = (0.0, 100.0)
cog_range = (0.0, 360.0)

input_scaler = MinMaxScaler()
input_scaler.min_ = np.array([
    -lat_range[0] / (lat_range[1] - lat_range[0]),
    -lon_range[0] / (lon_range[1] - lon_range[0]),
    -sog_range[0] / (sog_range[1] - sog_range[0]),
    -cog_range[0] / (cog_range[1] - cog_range[0])
])
input_scaler.scale_ = np.array([
    1 / (lat_range[1] - lat_range[0]),
    1 / (lon_range[1] - lon_range[0]),
    1 / (sog_range[1] - sog_range[0]),
    1 / (cog_range[1] - cog_range[0])
])
input_scaler.feature_names_in_ = np.array(['위도', '경도', 'SOG', 'COG'])
# --------------------------------------------------------------------------------

# 목적지 좌표 설정 - 포항
pohang = (36.0320, 129.3884)
donghae = (37.5340, 129.1161)
ulsan = (35.4985, 129.3850)
mokpo  = (34.7925, 126.3814)
jeju = (33.5136, 126.5230)

dest_lat = ulsan[0]
dest_lon = ulsan[1]

# 입력 시퀀스 생성
test_input_seq = []

for i, (_, row) in enumerate(initial_seq.iterrows()):
    # 1. 정규화된 위도, 경도, SOG, COG
    scaled = input_scaler.transform([[row['위도'], row['경도'], row['SOG'], row['COG']]])[0]
    
    # 2. 목적지 위도, 경도 정규화
    dest_scaled = input_scaler.transform([[dest_lat, dest_lon, 0, 0]])[0]
    dest_lat_scaled = dest_scaled[0]
    dest_lon_scaled = dest_scaled[1]

    # 3. Δlat, Δlon (정규화 x)
    delta_lat = dest_lat - row['위도']
    delta_lon = dest_lon - row['경도']
    
    # 4. 거리
    distance = np.sqrt(delta_lat ** 2 + delta_lon ** 2)

    # 최종 특성 벡터
    input_row = list(scaled) + [dest_lat_scaled, dest_lon_scaled, distance]
    test_input_seq.append(input_row)

# 7. 텐서로 변환: (1, seq_len, 7)
test_input_seq = torch.tensor([test_input_seq], dtype=torch.float32)

In [ ]:
def inverse_transform_preds(preds, scaler):
    """
    역정규화를 수행하여 원래의 값으로 변환
    preds: 예측된 값들 (numpy 배열), shape: (steps, output_size)
    scaler: 학습에 사용된 MinMaxScaler
    """
    # 위도, 경도, SOG, COG 값만 역정규화
    preds_unscaled = preds.copy()  # 예측된 값을 복사

    # 위도, 경도, SOG, COG를 역정규화
    preds_unscaled[:, :4] = scaler.inverse_transform(preds_unscaled[:, :4])  # 역정규화
    return preds_unscaled

import folium
from folium.plugins import AntPath

def visualize_route(initial_seq, preds_inverse, dest_lat, dest_lon, scaler):
    """
    예측된 경로를 시각화하는 함수
    initial_seq: 초기 입력 시퀀스 (numpy 배열), shape: (10, 6)
    preds_inverse: 역정규화된 예측 결과, shape: (steps, 4)
    dest_lat, dest_lon: 목적지 좌표
    """
    # 초기 위치
    start = initial_seq[0, 0][:4]
    start = scaler.inverse_transform([start])
    start_lat = start[0][0] 
    start_lon = start[0][1]
    # 지도 생성 (출발지와 목적지가 모두 보이도록 설정)
    route_map = folium.Map(location=[start_lat, start_lon], zoom_start=6)

    # 시작점, 목적지 마커 추가
    folium.Marker([start_lat, start_lon], tooltip='Start', icon=folium.Icon(color='green')).add_to(route_map)
    folium.Marker([dest_lat, dest_lon], tooltip='Destination', icon=folium.Icon(color='red')).add_to(route_map)

    # 예측 경로
    route_coords = [[lat, lon] for lat, lon in preds_inverse[:, :2]]  # 위도, 경도만 사용
    # 예측 경로를 PolyLine으로 시각화
    folium.PolyLine(route_coords, color='blue', weight=3, tooltip="Predicted Route").add_to(route_map)

    # 예측 경로에 애니메이션 효과 추가
    AntPath(route_coords).add_to(route_map)

    return route_map    

# 예측 결과를 역정규화 후 시각화하는 전체 코드
def predict_and_visualize(model, initial_seq, dest_lat, dest_lon, scaler, max_steps=150, distance_threshold=0.05):
    preds = predict_autoregressive(model, initial_seq, dest_lat, dest_lon, scaler, max_steps, distance_threshold)
    #preds = predict_autoregressive_mc_dropout(model, initial_seq, dest_lat, dest_lon, scaler, max_steps, distance_threshold, num_mc_samples=10) 
    #preds = preds.squeeze(1)
    preds_inverse = inverse_transform_preds(preds, scaler)
    # 예측 경로 시각화
    route_map = visualize_route(initial_seq.numpy(), preds_inverse, dest_lat, dest_lon, scaler)

    return route_map, preds_inverse

# 기 시퀀스와 목적지 좌표로 예측 및 시각화
route_map, predictions = predict_and_visualize(model, test_input_seq, dest_lat=dest_lat, dest_lon=dest_lon, scaler=input_scaler)
route_map.save('predicted_route_map_v2.html')